# JPMorgan Chase (JPM): Balance of Power

Companion to the [Bubble Analytics article](https://www.bubble-analytics.com/blog/JPMorgan-Chase-JPM-Balance-of-Power-2026-10-14), dated 2026-10-14.

Run all cells from this folder after installing `requirements.txt` and JupyterLab.
The included frozen data covers 2016-10-14 to 2026-10-13.
The measured backtest starts on 2016-10-14 after any indicator warmup.
Data files are read locally; no account, API key or network connection is needed to run the backtest.
`TEMPLATE.ipynb` is the existing general strategy template and is separate from this completed example.

The account starts with $100,000. Fees are 0.05% on entry and 0.05% on exit.
Signals are assessed after the close; market entries execute at the next session's open.
Stops and targets, where present, follow the code below. Daily bars cannot establish every intraday price sequence.
Open positions remain marked to market at the end of the sample; closed-trade statistics exclude them.
Spreads, additional slippage, financing and taxes are not modelled.
The benchmark uses the same sample and price convention as the article.

These are historical, in-sample results, not a promise of future performance.


In [ ]:
import hashlib
from pathlib import Path
input_hashes = {'JPM-daily.csv.gz': '049161bfd24c2339e9f8d9c63c7ee6bf5bd7a64306e85da38f7b449d0af4d2e4'}
for filename, expected_hash in input_hashes.items():
    assert hashlib.sha256((Path('data')/filename).read_bytes()).hexdigest() == expected_hash, 'Input file changed: ' + filename


## Strategy and backtest


In [ ]:
from IPython.display import display
"""Reproduce the JPM notebook strategy against frozen adjusted daily prices."""
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import backtesting
from backtesting import Backtest, Strategy
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday,
    sunday_to_monday, USMartinLutherKingJr, USPresidentsDay, GoodFriday,
    USMemorialDay, USLaborDay, USThanksgivingDay)

ROOT = Path.cwd()
(ROOT/'snapshots').mkdir(exist_ok=True)
(ROOT/'charts').mkdir(exist_ok=True)

raw = pd.read_csv(ROOT/'data/JPM-daily.csv.gz', parse_dates=['Date'], float_precision='round_trip').set_index('Date')
assert raw.index.is_monotonic_increasing and not raw.index.has_duplicates
assert str(raw.index[0].date()) == '2016-10-14'
assert str(raw.index[-1].date()) == '2026-10-13'
assert np.isfinite(raw.to_numpy()).all()
assert (raw[['Open','High','Low','Close']] > 0).all().all()
assert (raw.High >= raw[['Open','Close','Low']].max(axis=1)).all()
assert (raw.Low <= raw[['Open','Close','High']].min(axis=1)).all()
class EquityHolidays(AbstractHolidayCalendar):
    rules = [Holiday('New Year', month=1, day=1, observance=sunday_to_monday),
             USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay,
             Holiday('Juneteenth', month=6, day=19, start_date='2022-01-01', observance=nearest_workday),
             Holiday('Independence', month=7, day=4, observance=nearest_workday),
             USLaborDay, USThanksgivingDay,
             Holiday('Christmas', month=12, day=25, observance=nearest_workday)]
holidays = EquityHolidays().holidays(raw.index[0], raw.index[-1]).union(pd.DatetimeIndex(['2018-12-05','2025-01-09']))
expected = pd.bdate_range(raw.index[0], raw.index[-1]).difference(holidays)
assert raw.index.equals(expected.rename('Date')), 'Incomplete trading session sample'

def indicators(frame):
    data = frame.copy()
    assert data.High.ne(data.Low).all(), 'Zero daily range requires review'
    data['BOP'] = (data.Close-data.Open)/(data.High-data.Low)
    data['SBOP'] = data.BOP.rolling(17).mean()
    data['histo'] = (data.BOP-data.SBOP).rolling(17).mean().rolling(8).mean()
    data['signal'] = np.select([
        (data.histo<0)&(data.histo>data.histo.shift(1)),
        (data.histo.shift(1)>0)&(data.histo<0),
        (data.SBOP.shift(1)<-0.1)&(data.SBOP>-0.1)], [1,2,3], default=0)
    return data

data=indicators(raw)
for end in (17,25,41,200,1000,len(raw)-1):
    pd.testing.assert_frame_equal(indicators(raw.iloc[:end]),data.iloc[:end])
assert data.signal.iloc[0]==0
class JPMBOP(Strategy):
    def init(self): self.signal=self.I(lambda: self.data.signal)
    def next(self):
        price=float(self.data.Close[-1])
        signal=int(self.signal[-1])
        if signal==1:
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99,sl=0.94*price,tp=1.03*price,tag=1)
        elif signal==3:
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99,sl=0.92*price,tp=1.035*price,tag=3)
        elif signal==2:
            if self.position.is_long or not self.position:
                self.position.close()

bt=Backtest(data,JPMBOP,cash=100000,margin=1,exclusive_orders=True,commission=0.0005,
            trade_on_close=False,finalize_trades=False)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    stats=bt.run()
trades=stats['_trades'].copy()
equity=stats['_equity_curve'].copy()
opened=stats['_strategy'].trades
benchmark=(100000*data.Close/data.Close.iloc[0]).rename('BenchmarkEquity')
assert backtesting.__version__=='0.6.5'
fees=trades.Size.abs()*(trades.EntryPrice+trades.ExitPrice)*0.0005
assert np.allclose(trades.Commission,fees)
assert np.allclose(trades.PnL,trades.Size*(trades.ExitPrice-trades.EntryPrice)-fees)
assert np.allclose(trades.ReturnPct,trades.PnL/(trades.Size*trades.EntryPrice))
assert np.isclose((trades.PnL>0).mean()*100,stats['Win Rate [%]'])
assert np.isclose((equity.Equity.iloc[-1]/100000-1)*100,stats['Return [%]'])
assert np.isclose((benchmark.iloc[-1]/100000-1)*100,stats['Buy & Hold Return [%]'])
assert np.isclose((equity.Equity/equity.Equity.cummax()-1).min()*100,stats['Max. Drawdown [%]'])
assert len(trades)==stats['# Trades']
assert trades.Size.gt(0).all()
trades['SignalClose']=[data.Close.iloc[int(i)-1] for i in trades.EntryBar]
trades['InitialStop']=trades.SignalClose*np.where(trades.Tag==1,.94,.92)
trades['InitialTarget']=trades.SignalClose*np.where(trades.Tag==1,1.03,1.035)
exit_kinds=[]
for t in trades.itertuples():
    assert data.signal.iloc[t.EntryBar-1]==t.Tag
    assert np.isclose(t.EntryPrice,data.Open.iloc[t.EntryBar])
    assert np.isclose(t.TP,t.InitialTarget)
    # Market close orders take priority over contingent orders at that next open.
    if data.signal.iloc[t.ExitBar-1]==2 and t.ExitBar>t.EntryBar:
        assert np.isclose(t.ExitPrice,data.Open.iloc[t.ExitBar])
        exit_kinds.append('Sell signal')
    elif data.Low.iloc[t.ExitBar]<=t.InitialStop:
        assert np.isclose(t.ExitPrice,min(data.Open.iloc[t.ExitBar],t.InitialStop))
        exit_kinds.append('Stop')
    else:
        assert data.High.iloc[t.ExitBar]>=t.InitialTarget
        assert np.isclose(t.ExitPrice,max(data.Open.iloc[t.ExitBar],t.InitialTarget))
        exit_kinds.append('Target')
trades['ExitKind']=exit_kinds
exposure=np.zeros(len(data),dtype=bool)
reconstructed=pd.Series(100000.,index=data.index)
for t in trades.itertuples():
    exposure[t.EntryBar:t.ExitBar+1]=True
    live=(data.index>=t.EntryTime)&(data.index<t.ExitTime)
    reconstructed.loc[live]+=t.Size*(data.Close.loc[live]-t.EntryPrice)-t.Size*t.EntryPrice*.0005
    reconstructed.loc[data.index>=t.ExitTime]+=t.PnL
for t in opened:
    exposure[t.entry_bar:]=True
    live=data.index>=t.entry_time
    reconstructed.loc[live]+=t.size*(data.Close.loc[live]-t.entry_price)-t.size*t.entry_price*.0005
assert np.allclose(reconstructed,equity.Equity)
closed_exposure=np.zeros(len(data),dtype=bool)
for t in trades.itertuples():closed_exposure[t.EntryBar:t.ExitBar+1]=True
assert np.isclose(closed_exposure.mean()*100,stats['Exposure Time [%]'])
trades['CumulativePnL']=trades.PnL.cumsum()
realised=pd.Series(100000.,index=data.index)
for t in trades.itertuples():realised.loc[data.index>=t.ExitTime]+=t.PnL
realised_dd=(realised/realised.cummax()-1)*100
metrics={str(k):str(v) if isinstance(v,(pd.Timestamp,pd.Timedelta)) else float(v)
         for k,v in stats.items() if not str(k).startswith('_') and isinstance(v,(int,float,np.number,pd.Timestamp,pd.Timedelta))}
metrics.update(symbol='JPM',company='JPMorgan Chase & Co.',initialCash=100000,commissionPerSide=.0005,
    dataFirst=str(raw.index[0].date()),dataLast=str(raw.index[-1].date()),rawRows=len(raw),measuredRows=len(data),
    sbopFirstValid=str(data.SBOP.first_valid_index().date()),histogramFirstValid=str(data.histo.first_valid_index().date()),
    firstPossibleSBOPCross=str(data.index[17].date()),firstPossibleHistogramComparison=str(data.index[40].date()),
    firstSignal=str(data.index[data.signal.ne(0)][0].date()),signalCounts={str(k):int(v) for k,v in data.signal.value_counts().items()},
    benchmarkMaxDrawdownPct=float((benchmark/benchmark.cummax()-1).min()*100),benchmarkFinalEquity=float(benchmark.iloc[-1]),
    closedTradePnL=float(trades.PnL.sum()),closedTradeFees=float(fees.sum()),openTradesAtEnd=len(opened),
    winningTrades=int(trades.PnL.gt(0).sum()),losingTrades=int(trades.PnL.lt(0).sum()),
    meanWinPct=float(trades.loc[trades.PnL>0,'ReturnPct'].mean()*100),meanLossPct=float(trades.loc[trades.PnL<0,'ReturnPct'].mean()*100),
    realisedMaxDrawdownPct=float(realised_dd.min()),exitCounts=trades.ExitKind.value_counts().to_dict(),
    entryTypeCounts={str(k):int(v) for k,v in trades.Tag.value_counts().items()},
    randomSeed=20261014,randomisation='100 permutations of summed net trade returns, without compounding',summedTradeReturnPct=float(trades.ReturnPct.sum()*100),
    openTrades=[dict(entry=str(t.entry_time),entryPrice=float(t.entry_price),size=int(t.size),stop=t.sl,target=t.tp) for t in opened],
    pendingOrdersAtEnd=len(stats['_strategy'].orders),backtestingVersion=backtesting.__version__,warnings=[str(w.message) for w in captured],
    snapshotSha256=hashlib.sha256((ROOT/'data/JPM-daily.csv.gz').read_bytes()).hexdigest(),
    verification=dict(sessionCoverage=True,prefixCausality=True,fees=True,ledgerReturns=True,winRate=True,dailyEquityReconstruction=True,benchmark=True,drawdown=True,exposure=True,entryRules=True,exitRules=True))
metrics['engineClosedTradeExposurePct']=metrics['Exposure Time [%]']
metrics['Exposure Time [%]']=float(exposure.mean()*100)
metrics['engineCAGRCalendar252Pct']=metrics['CAGR [%]']
metrics['CAGR [%]']=float(((equity.Equity.iloc[-1]/100000)**(365.2425/(data.index[-1]-data.index[0]).days)-1)*100)
metrics['openTradeEntryFees']=float(sum(t.size*t.entry_price*.0005 for t in opened))
metrics['openTradeGrossPnL']=float(sum(t.size*(data.Close.iloc[-1]-t.entry_price) for t in opened))
metrics['openTradeNetPnL']=metrics['openTradeGrossPnL']-metrics['openTradeEntryFees']
assert np.isclose(100000+metrics['closedTradePnL']+metrics['openTradeNetPnL'],metrics['Equity Final [$]'])
metrics['gapAboveTargetEntryCount']=int((trades.EntryPrice>=trades.InitialTarget).sum())
metrics['sameDayTrades']=int((trades.EntryBar==trades.ExitBar).sum())
metrics['gapBelowStopExits']=int(((trades.ExitKind=='Stop')&(trades.ExitPrice<trades.InitialStop-1e-8)).sum())
metrics['bothLevelsTouchedExitCount']=int(sum(data.High.iloc[t.ExitBar]>=t.InitialTarget and data.Low.iloc[t.ExitBar]<=t.InitialStop for t in trades.itertuples()))
metrics['verification']['openPositionReconciled']=True
rng=np.random.default_rng(20261014)
curves=np.array([np.r_[0,np.cumsum(rng.permutation(trades.ReturnPct.to_numpy())*100)] for _ in range(100)])
assert np.allclose(curves[:,-1],metrics['summedTradeReturnPct'])
np.save(ROOT/'snapshots/trade-order-curves.npy',curves)
trades.to_csv(ROOT/'snapshots/trades.csv',index=False)
equity.to_csv(ROOT/'snapshots/equity.csv')
data.to_csv(ROOT/'snapshots/indicators.csv')
benchmark.to_csv(ROOT/'snapshots/benchmark.csv')
realised.rename('RealisedBalance').to_csv(ROOT/'snapshots/realised-balance.csv')
(ROOT/'snapshots/results.json').write_text(json.dumps(metrics,indent=2,allow_nan=False)+'\n')
display(pd.Series({k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}, name='Result'))


## Charts and account results


In [ ]:
"""Keep the source notebook's Plotly design and regenerate verified values."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
(ROOT/'charts').mkdir(exist_ok=True)
m = json.loads((ROOT/'snapshots/results.json').read_text())
d = pd.read_csv(ROOT/'snapshots/indicators.csv',index_col='Date',parse_dates=True)
t = pd.read_csv(ROOT/'snapshots/trades.csv',parse_dates=['EntryTime','ExitTime'])
e = pd.read_csv(ROOT/'snapshots/equity.csv',index_col=0,parse_dates=True)
b = pd.read_csv(ROOT/'snapshots/benchmark.csv',index_col=0,parse_dates=True).iloc[:,0]
curves = np.load(ROOT/'snapshots/trade-order-curves.npy')
CHARTS = {}

def themed(fig,title,yaxis):
    fig.update_layout(title=title,template='plotly_dark',autosize=False,width=1100,height=700,
        yaxis_title=yaxis,font=dict(family='Arial, Helvetica, sans-serif',size=14),
        margin=dict(l=95,r=45,t=85,b=80),legend=dict(orientation='h',y=1.03,x=0))
    fig.update_yaxes(gridcolor='#171717')
    fig.update_xaxes(gridcolor='#171717')
    return fig

window=d.iloc[2200:2550]
f=make_subplots(rows=3,cols=1,shared_xaxes=True,row_heights=[.50,.25,.25],vertical_spacing=.05)
f.add_trace(go.Candlestick(x=window.index,open=window.Open,high=window.High,low=window.Low,close=window.Close,
    increasing_line_color='rgba(19,156,19,0.8)',decreasing_line_color='rgba(175,07,49,0.8)',name='JPM'),row=1,col=1)
for signal,colour,label in [(1,'white','Buy: rising histogram'),(3,'gold','Buy: BOP recovery'),(2,'red','Close long signal')]:
    markers=np.where(window.signal==signal,window.High*1.002 if signal==2 else window.Low*.998,np.nan)
    f.add_trace(go.Scatter(x=window.index,y=markers,mode='markers',marker=dict(size=7,color=colour),name=label),row=1,col=1)
f.add_trace(go.Scatter(x=window.index,y=window.SBOP,line=dict(color='#00DF9A',width=2),name='Smoothed Balance of Power',showlegend=False),row=2,col=1)
f.add_trace(go.Bar(x=window.index,y=window.histo,marker_color='gray',name='BOP histogram'),row=3,col=1)
f.update_layout(template='plotly_dark',width=1100,height=800,autosize=False,xaxis_rangeslider_visible=False,
    font=dict(family='Arial, Helvetica, sans-serif',size=13),margin=dict(l=75,r=205,t=60,b=65),
    annotations=[dict(text=text,font=dict(color='white',size=12),xref='paper',yref='paper',x=.02,y=y,showarrow=False)
       for text,y in [('JPM Daily Chart',1.03),('Balance of Power Indicator',.49),('Balance of Power Histogram',.22)]],
    legend=dict(x=1.02,y=1,font=dict(size=11)))
f.update_yaxes(gridcolor='#171717');f.update_xaxes(gridcolor='#171717',rangebreaks=[dict(bounds=['sat','mon'])])
f.update_yaxes(title_text='Adjusted price ($)',row=1,col=1)
CHARTS['chart-bop']=f

f=go.Figure()
f.add_trace(go.Scatter(x=e.index,y=(e.Equity/100000-1)*100,name='BOP strategy after fees',line=dict(color='#00df9a',width=2)))
f.add_trace(go.Scatter(x=b.index,y=(b/100000-1)*100,name='JPM buy and hold',line=dict(color='white',width=1.5)))
CHARTS['chart-equity']=themed(f,'JPM BOP Strategy · Account Returns','Cumulative return (%)')
f=go.Figure()
f.add_trace(go.Scatter(x=e.index,y=(e.Equity/e.Equity.cummax()-1)*100,name='BOP strategy',line=dict(color='#00df9a',width=2)))
f.add_trace(go.Scatter(x=b.index,y=(b/b.cummax()-1)*100,name='JPM buy and hold',line=dict(color='white',width=1.5)))
r=pd.read_csv(ROOT/'snapshots/realised-balance.csv',index_col=0,parse_dates=True).iloc[:,0]
f.add_trace(go.Scatter(x=r.index,y=(r/r.cummax()-1)*100,name='Closed trade balance',line=dict(color='gold',width=1.2)))
CHARTS['chart-drawdown']=themed(f,'JPM BOP Strategy · Daily Closing Drawdowns','Decline from previous peak (%)')
f=go.Figure(go.Scatter(x=[d.index[0],*t.ExitTime],y=[0,*t.CumulativePnL],mode='lines',name='Cumulative PnL',line=dict(color='#00df9a',shape='hv')))
CHARTS['chart-pnl']=themed(f,'JPM BOP Strategy · Closed Trade Profit','Cumulative realised profit ($)')
f=go.Figure()
for curve in curves:
    f.add_trace(go.Scatter(y=curve,mode='lines',opacity=0.6,showlegend=False))
CHARTS['chart-trade-order']=themed(f,'JPM BOP Strategy · 100 Randomised Trade Orders','Sum of net trade returns (percentage points)')
f.update_xaxes(title_text='Closed trade number')
f.add_annotation(text='Fixed seed · Same 146 closed trades · Summed percentages, without compounding',xref='paper',yref='paper',x=0,y=1.04,showarrow=False,font=dict(size=13,color='#aaaaaa'))

rows=[('Closed trades',str(int(m['# Trades']))),('Exposure time',f"{m['Exposure Time [%]']:.2f}%"),
      ('Win rate',f"{m['Win Rate [%]']:.2f}%"),('Strategy return',f"{m['Return [%]']:.2f}%"),
      ('Buy and hold return',f"{m['Buy & Hold Return [%]']:.2f}%"),('Account maximum drawdown',f"{m['Max. Drawdown [%]']:.2f}%"),
      ('Realised maximum drawdown',f"{m['realisedMaxDrawdownPct']:.2f}%"),
      ('Buy and hold max drawdown',f"{m['benchmarkMaxDrawdownPct']:.2f}%"),('Sharpe ratio',f"{m['Sharpe Ratio']:.2f}"),
      ('Sortino ratio',f"{m['Sortino Ratio']:.2f}"),('Calmar ratio',f"{m['Calmar Ratio']:.2f}"),
      ('Best trade',f"{m['Best Trade [%]']:.2f}%"),('Worst trade',f"{m['Worst Trade [%]']:.2f}%"),
      ('Average trade (geometric)',f"{m['Avg. Trade [%]']:.2f}%"),('Profit factor',f"{m['Profit Factor']:.2f}"),
      ('Expectancy (arithmetic)',f"{m['Expectancy [%]']:.2f}%")]
stripe=['#141A1E' if i%2==0 else '#101519' for i in range(len(rows))]
f=go.Figure(go.Table(columnwidth=[1.6,1],header=dict(values=['',''],height=1,fill_color='#161D21',line_color='#161D21',font=dict(size=1,color='#161D21')),
    cells=dict(values=list(zip(*rows)),align=['left','right'],height=34,font=dict(size=13,color='#E6ECEF',family='Arial, Helvetica, sans-serif'),
    fill_color=[stripe,stripe],line_color='rgba(255,255,255,0.08)',line_width=1)))
f.update_layout(width=500,height=105+34*len(rows)+35,paper_bgcolor='#12181B',plot_bgcolor='#161D21',margin=dict(l=30,r=30,t=105,b=35))
f.add_annotation(text='<b>JPM · Balance of Power<br>Strategy</b>',x=0,y=1.15,xref='paper',yref='paper',xanchor='left',showarrow=False,align='left',font=dict(size=12,color='#E6ECEF',family='Arial, Helvetica, sans-serif'))
f.add_annotation(text="<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>",x=1,y=1.15,xref='paper',yref='paper',xanchor='right',showarrow=False,font=dict(size=12,family='Arial, Helvetica, sans-serif'))
f.add_shape(type='rect',xref='paper',yref='paper',x0=0,y0=0,x1=1,y1=1,line=dict(color='rgba(255,255,255,0.08)',width=1),fillcolor='#161D21',layer='below')
CHARTS['stats-summary']=f

# Consistent branding with a dedicated header above titles and legends.
wordmark = "<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>"
for figure in CHARTS.values():
    figure.layout.annotations = tuple(a for a in figure.layout.annotations if 'BUBBLE' not in (a.text or ''))
    width, height = figure.layout.width or 1100, figure.layout.height or 700
    figure.update_layout(width=width, height=height, margin_t=max(figure.layout.margin.t or 0, 150))
    if figure.layout.title.text:
        figure.update_layout(title_y=1-65/height, title_yanchor='top')
    inner_height = height - figure.layout.margin.t - (figure.layout.margin.b or 80)
    figure.add_annotation(text=wordmark, x=1, y=1+(figure.layout.margin.t-20)/inner_height,
        xref='paper', yref='paper', xanchor='right', yanchor='top', showarrow=False,
        font=dict(size=12, family='Arial, Helvetica, sans-serif'))


In [ ]:
for figure in CHARTS.values():
    display(figure)
